# Module D: Constructor Efficiency

**F1 Race Intelligence Project**

Raw points only tell part of the story. A team that scores 500 points with the fastest car
may be underperforming, while a midfield team overachieving with clever strategy might be
the most efficient operation on the grid.

This module computes **efficiency ratings** that compare actual points scored against
expected points based on qualifying position and grid-to-points conversion rates.

**Analyses performed:**
1. Actual vs Expected points per constructor
2. Efficiency rating (over/under performance percentage)
3. Teammate gap analysis
4. DNF rate and reliability analysis
5. Rolling form trends
6. Net positions gained/lost per constructor

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.viz import (
    efficiency_bar_chart, efficiency_rating_chart, f1_layout,
    F1_RED, F1_WHITE, F1_BG, TEAM_COLORS
)

DB_PATH = '../data/processed/f1.db'
con = duckdb.connect(DB_PATH, read_only=True)

RACE_YEAR = 2024
print(f'Connected to DuckDB — analyzing {RACE_YEAR} constructor efficiency')

## 1. Grid-to-Points Conversion Baseline

Before computing efficiency, we need a baseline: **how many points does each grid position
historically convert into?** This accounts for the fact that P1 on the grid does not always
win, and that midfield positions have variable point yields.

In [ ]:
# Grid to points lookup table
grid_points = con.execute("""
    SELECT * FROM grid_to_points_lookup
    ORDER BY grid
""").fetchdf()

if not grid_points.empty:
    print(f'Grid positions with data: {len(grid_points)}')
    display(grid_points.head(10))
    
    fig = go.Figure(go.Bar(
        x=grid_points['grid'],
        y=grid_points['expected_points'],
        marker_color=F1_RED,
        text=grid_points['expected_points'].round(1),
        textposition='outside',
    ))
    fig = f1_layout(fig, 'Historical Average Points by Grid Position', height=400)
    fig.update_xaxes(title_text='Grid Position', dtick=1)
    fig.update_yaxes(title_text='Average Points Scored')
    fig.show()
    
    print(f'\nP1 avg points: {grid_points.iloc[0]["expected_points"]:.1f}')
    print(f'P10 avg points: {grid_points[grid_points["grid"]==10]["expected_points"].values[0]:.1f}')
    print(f'P20 avg points: {grid_points[grid_points["grid"]==20]["expected_points"].values[0]:.1f}')

## 2. Actual vs Expected Points

For each constructor-season, we compare their actual points against what their qualifying
positions would predict. Teams scoring above expected are "over-performing" through
superior race execution, strategy, and reliability.

In [ ]:
# Constructor efficiency data
eff_df = con.execute("""
    SELECT ce.*, c.constructor_name
    FROM constructor_efficiency ce
    JOIN constructors c ON ce.constructor_id = c.constructor_id
    ORDER BY year DESC, actual_points DESC
""").fetchdf()

print(f'Constructor efficiency records: {len(eff_df)}')
print(f'Years covered: {eff_df["year"].min()} - {eff_df["year"].max()}')

# Show 2024 season
eff_2024 = eff_df[eff_df['year'] == RACE_YEAR].sort_values('efficiency_rating', ascending=False)
print(f'\n{RACE_YEAR} Constructor Efficiency:')
display(eff_2024[['constructor_name', 'actual_points', 'expected_points',
                   'points_vs_expectation', 'efficiency_rating', 'dnf_rate']].round(1))

In [ ]:
# Actual vs Expected points chart for 2024
if not eff_df.empty:
    target_year = eff_df['year'].max()
    fig = efficiency_bar_chart(eff_df, target_year)
    fig.show()
else:
    print('No efficiency data available')

## 3. Efficiency Rating

The **efficiency rating** is: `(actual_points / expected_points) * 100`

- Rating > 100: overperforming (getting more points than qualifying position predicts)
- Rating = 100: performing exactly as expected
- Rating < 100: underperforming (losing points relative to grid position)

In [ ]:
# Efficiency rating chart
if not eff_df.empty:
    target_year = eff_df['year'].max()
    fig = efficiency_rating_chart(eff_df, target_year)
    fig.show()
    
    # Print notable over/under performers
    year_eff = eff_df[eff_df['year'] == target_year].copy()
    if 'efficiency_rating' in year_eff.columns:
        overperformers = year_eff[year_eff['efficiency_rating'] > 105].sort_values('efficiency_rating', ascending=False)
        underperformers = year_eff[year_eff['efficiency_rating'] < 95].sort_values('efficiency_rating')
        
        if not overperformers.empty:
            print(f'\n{target_year} Overperformers (>105%):')
            for _, row in overperformers.iterrows():
                print(f'  {row["constructor_id"]}: {row["efficiency_rating"]:.1f}%')
        
        if not underperformers.empty:
            print(f'\n{target_year} Underperformers (<95%):')
            for _, row in underperformers.iterrows():
                print(f'  {row["constructor_id"]}: {row["efficiency_rating"]:.1f}%')

## 4. Efficiency Trends Over Multiple Seasons

Tracking efficiency across seasons reveals which teams are improving their operations
and which are in decline.

In [ ]:
# Multi-year efficiency trends for key teams
if 'efficiency_rating' in eff_df.columns:
    key_teams = ['red_bull', 'ferrari', 'mercedes', 'mclaren']
    available_teams = eff_df[eff_df['constructor_id'].isin(key_teams)]['constructor_id'].unique()
    
    if len(available_teams) == 0:
        # Use top 4 teams by total points
        available_teams = eff_df.groupby('constructor_id')['actual_points'].sum().nlargest(4).index.tolist()
    
    fig = go.Figure()
    for team in available_teams:
        team_data = eff_df[eff_df['constructor_id'] == team].sort_values('year')
        fig.add_trace(go.Scatter(
            x=team_data['year'],
            y=team_data['efficiency_rating'],
            name=team,
            mode='lines+markers',
            line=dict(color=TEAM_COLORS.get(team, '#666'), width=2),
            marker=dict(size=6)
        ))
    
    fig = f1_layout(fig, 'Constructor Efficiency Rating Trends', height=450)
    fig.add_hline(y=100, line_dash='dash', line_color='#666', annotation_text='Expected (100%)')
    fig.update_yaxes(title_text='Efficiency Rating (%)')
    fig.update_xaxes(title_text='Season')
    fig.show()

## 5. Teammate Gap Analysis

Comparing teammates provides a fair benchmark since both drivers share the same car.
A large teammate gap indicates one driver is extracting significantly more performance.

In [ ]:
# Teammate gap analysis: qualifying and race position gaps + season points
teammate_summary = con.execute(f"""
    SELECT
        tg.constructor_id,
        d1.full_name AS driver_1,
        d2.full_name AS driver_2,
        ROUND(AVG(tg.quali_gap_positions), 2) AS avg_quali_gap,
        ROUND(AVG(tg.race_gap_positions), 2) AS avg_race_gap,
        COUNT(*) AS races_together
    FROM teammate_gap tg
    JOIN drivers d1 ON tg.driver_1 = d1.driver_id
    JOIN drivers d2 ON tg.driver_2 = d2.driver_id
    WHERE tg.year = {RACE_YEAR}
    GROUP BY tg.constructor_id, d1.full_name, d2.full_name
    ORDER BY avg_quali_gap DESC
""").fetchdf()

# Also get season points per driver for the gap
driver_points = con.execute(f"""
    SELECT r.constructor_id, d.full_name, SUM(r.points) AS season_points
    FROM results r
    JOIN drivers d ON r.driver_id = d.driver_id
    WHERE r.year = {RACE_YEAR}
    GROUP BY r.constructor_id, d.full_name
""").fetchdf()

if not teammate_summary.empty:
    print(f'{RACE_YEAR} Teammate Gaps ({len(teammate_summary)} pairings):')
    display(teammate_summary)
    
    # Qualifying gap bar chart
    fig = go.Figure(go.Bar(
        x=teammate_summary['constructor_id'],
        y=teammate_summary['avg_quali_gap'],
        marker_color=[TEAM_COLORS.get(c, '#666') for c in teammate_summary['constructor_id']],
        text=teammate_summary['avg_quali_gap'].round(2),
        textposition='outside',
    ))
    fig = f1_layout(fig, f'{RACE_YEAR} Average Teammate Qualifying Gap (positions)', height=450)
    fig.update_xaxes(tickangle=-45)
    fig.update_yaxes(title_text='Avg Qualifying Gap (positions)')
    fig.show()
    
    # Points gap per team (driver 1 vs driver 2)
    if not driver_points.empty:
        team_pts = driver_points.groupby('constructor_id').apply(
            lambda g: g.nlargest(2, 'season_points') if len(g) >= 2 else g
        ).reset_index(drop=True)
        
        fig = go.Figure()
        for team in team_pts['constructor_id'].unique():
            tdata = team_pts[team_pts['constructor_id'] == team].sort_values('season_points', ascending=False)
            if len(tdata) >= 2:
                fig.add_trace(go.Bar(
                    x=[team], y=[tdata.iloc[0]['season_points']],
                    name=tdata.iloc[0]['full_name'],
                    marker_color=TEAM_COLORS.get(team, '#666'),
                    showlegend=False,
                ))
                fig.add_trace(go.Bar(
                    x=[team], y=[tdata.iloc[1]['season_points']],
                    name=tdata.iloc[1]['full_name'],
                    marker_color=TEAM_COLORS.get(team, '#666'),
                    opacity=0.5,
                    showlegend=False,
                ))
        fig = f1_layout(fig, f'{RACE_YEAR} Teammate Season Points Comparison', height=450)
        fig.update_layout(barmode='group')
        fig.update_yaxes(title_text='Season Points')
        fig.update_xaxes(tickangle=-45)
        fig.show()

## 6. DNF Rate & Reliability

Reliability is a critical component of team efficiency. A fast but unreliable car
hemorrhages points. We analyze DNF (Did Not Finish) rates by constructor.

In [ ]:
# DNF analysis by constructor (2010-2024 era)
dnf_analysis = con.execute("""
    SELECT
        constructor_id,
        COUNT(*) AS total_entries,
        SUM(CASE WHEN position IS NULL THEN 1 ELSE 0 END) AS dnfs,
        ROUND(SUM(CASE WHEN position IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS dnf_pct
    FROM results
    WHERE year >= 2020
    GROUP BY constructor_id
    HAVING COUNT(*) >= 20
    ORDER BY dnf_pct DESC
""").fetchdf()

if not dnf_analysis.empty:
    fig = go.Figure(go.Bar(
        x=dnf_analysis['constructor_id'],
        y=dnf_analysis['dnf_pct'],
        marker_color=[TEAM_COLORS.get(c, '#666') for c in dnf_analysis['constructor_id']],
        text=dnf_analysis['dnf_pct'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
    ))
    fig = f1_layout(fig, 'DNF Rate by Constructor (2020-2024)', height=450)
    fig.update_yaxes(title_text='DNF Rate (%)')
    fig.update_xaxes(tickangle=-45)
    fig.show()
    
    print('DNF rates (2020-2024):')
    display(dnf_analysis)

## 7. Net Positions Gained

Net positions gained (finish position - grid position) captures race-day execution.
Positive values mean the team consistently finishes behind where they qualify;
negative values mean they are strong race-day converters.

In [ ]:
# Net positions gained — aggregate per constructor from the per-driver view
net_pos = con.execute(f"""
    SELECT
        constructor_id,
        COUNT(*) AS total_entries,
        ROUND(AVG(net_positions_gained), 2) AS avg_net_positions,
        SUM(CASE WHEN net_positions_gained > 0 THEN 1 ELSE 0 END) AS races_gained,
        SUM(CASE WHEN net_positions_gained < 0 THEN 1 ELSE 0 END) AS races_lost
    FROM net_positions
    WHERE year = {RACE_YEAR} AND net_positions_gained IS NOT NULL
    GROUP BY constructor_id
    ORDER BY avg_net_positions DESC
""").fetchdf()

if not net_pos.empty:
    fig = go.Figure(go.Bar(
        x=net_pos['avg_net_positions'],
        y=net_pos['constructor_id'],
        orientation='h',
        marker_color=['#44FF44' if v > 0 else '#FF4444' for v in net_pos['avg_net_positions']],
        text=net_pos['avg_net_positions'].round(2),
        textposition='outside',
    ))
    fig = f1_layout(fig, f'{RACE_YEAR} Average Net Positions Gained per Race', height=450)
    fig.update_xaxes(title_text='Net Positions (positive = gained places on race day)')
    fig.add_vline(x=0, line_dash='dash', line_color='#666')
    fig.show()
    
    print(f'{RACE_YEAR} Net positions by constructor:')
    display(net_pos)

## 8. Points Scored Per Race: Seasonal Consistency

Consistency is key in F1. We look at the variance in points scored to identify
which teams are steady scorers vs. boom-or-bust performers.

In [ ]:
# Points per race by constructor for the 2024 season
points_per_race = con.execute(f"""
    SELECT
        r.constructor_id,
        r.race_id,
        ra.race_name,
        SUM(r.points) AS race_points
    FROM results r
    JOIN races ra ON r.race_id = ra.race_id
    WHERE r.year = {RACE_YEAR}
    GROUP BY r.constructor_id, r.race_id, ra.race_name
    ORDER BY r.race_id
""").fetchdf()

if not points_per_race.empty:
    # Top 5 teams by total points
    top_teams = points_per_race.groupby('constructor_id')['race_points'].sum().nlargest(5).index.tolist()
    
    fig = go.Figure()
    for team in top_teams:
        team_data = points_per_race[points_per_race['constructor_id'] == team]
        fig.add_trace(go.Scatter(
            x=team_data['race_name'],
            y=team_data['race_points'],
            name=team,
            mode='lines+markers',
            line=dict(color=TEAM_COLORS.get(team, '#666'), width=2),
            marker=dict(size=5),
        ))
    fig = f1_layout(fig, f'{RACE_YEAR} Points per Race: Top 5 Constructors', height=450)
    fig.update_xaxes(tickangle=-45)
    fig.update_yaxes(title_text='Points')
    fig.show()
    
    # Consistency stats
    consistency = points_per_race.groupby('constructor_id')['race_points'].agg(['mean', 'std', 'sum'])
    consistency['cv'] = (consistency['std'] / consistency['mean'].replace(0, np.nan) * 100).round(1)
    consistency = consistency.sort_values('sum', ascending=False)
    print(f'\n{RACE_YEAR} Scoring consistency (lower CV% = more consistent):')
    display(consistency.head(10).round(1))

## Key Findings

1. **Efficiency ratings** reveal that raw points standings can be misleading. Teams with lower qualifying positions but strong race execution often outperform expectations.
2. **Teammate gaps** highlight internal team dynamics and driver quality.
3. **DNF rates** show that reliability alone can swing championship positions for midfield teams.
4. **Net positions gained** identifies teams with superior race-day conversion (strategy + tyre management + race craft).
5. **Scoring consistency** matters for championships: a team that reliably scores moderate points outperforms a team with high variance.

---
*Next: Module E (XGBoost Podium Predictor) combines all features into a machine learning model.*

In [ ]:
con.close()
print('Connection closed.')